# 제품별 페르소나 구축 노트북 (Pure Jupyter)

**입력 파일**
- `product.json`: 제품 정보 목록
- `people_segment.json`: 소비자 세그먼트 및 개인 정보 목록

**출력 파일**
- `/mnt/data/product_personas.json`: 전체 통합 JSON
- `/mnt/data/personas/<product>.json`: 제품별 개별 JSON
- `/mnt/data/personas/manifest.json`: 리스트
- `/mnt/data/product_personas_split.zip`: 제품별 JSON 묶음

**사용법**
1) 아래 `CONFIG` 셀 **한 곳만 수정**
2) 순서대로 실행 → 마지막 `run_pipeline()` 셀까지 실행


## 0) CONFIG — 여기만 고치면 됩니다 ✅

In [13]:
from pathlib import Path

CONFIG = {
    # ---- 파일 경로 ----
    "PATHS": {
        "DATA_DIR": "",
        "PRODUCTS": "product.json",
        "PEOPLE": "people_segment.json",
        "OUT_DIR": "/output",
        "PER_PRODUCT_DIRNAME": "personas",
        "SPLIT_ZIP": "product_personas_split.zip",
        "OUTPUT_JSON": "product_personas.json",
    },

    # ---- 생성 옵션 ----
    "TOP_K": 100,                 # 제품별 선택할 페르소나 수
    "ATTR_MIN": 10,             # attributes 최소 개수
    "ATTR_CAP": 14,             # attributes 상한

    # ---- 가격대 버킷 기준(KRW) ----
    "PRICE_BUCKET": {
        "LOW_MAX": 3000,        # ≤LOW_MAX → low
        "HIGH_MIN": 10000,      # ≥HIGH_MIN → high
    },

    # ---- 성향 점수 스케일 ----
    "ORI_SCALE_MAX": 9.0,       # orientations 최대값(보통 9점)

    # ---- 제품 특징 ↔ 성향 키 매핑 ----
    "FEATURE_TO_ORI": {
        "건강식품": "health",
        "고단백": "health",
        "프리미엄": "premium",
        "간편함": "convenience",
        "가성비": "price_sensitivity",
        "매콤한맛": "variety_seeking",
        "고소한맛": "taste",
        "감칠맛": "taste",
        "높은 만족도": None,
    },

    # ---- 카테고리 매핑(제품군 추론용 키워드) ----
    "CATEGORY_MAP": {
        "요거트": ["유가공품", "발효유", "우유", "요구르트"],
        "커피음료": ["커피및차", "커피", "커피음료"],
        "참치캔": ["조미수산가공품", "수산물통조림", "염건수산가공품"],
        "축산캔": ["축산캔", "육류가공품"],
        "참치액/조미료": ["조미식품", "장류", "조미소스", "조미료", "소스"],
        "유지류": ["유지류", "참기름", "들기름", "식용유"],
    },

    # ---- 구매결정요인(DC) 동의어/정규화 ----
    "DC_NORMALIZE": {
        "맛": ["맛"],
        "가격": ["가격"],
        "품질": ["품질"],
        "안전성": ["안전성", "영양(건강)", "영양"],
        "조리의 편리성": ["조리의 편리성"],
        "구입의 편리성": ["구입의 편리성"],
        "신선도": ["신선도 (제조일자, 소비(유통)기한 포함)", "신선도"],
    },

    # ---- 적합도 점수 계산 시 가산치 ----
    "FIT_WEIGHTS": {
        "TOP_CATEGORY_1": 1.0,
        "TOP_CATEGORY_2": 0.5,
        "HOUSEHOLD_MATCH": 1.0,
        "DC_CORE": 0.2,
        "DC_CONVENIENCE": 0.15,
        "DC_FRESHNESS": 0.1,
        "ORI_FEAT_MATCH": 1.0,
        "TASTE_FALLBACK": 0.2,
        "PRICE_LOW_PS_COEF": 0.5,
        "PRICE_LOW_BIAS": 0.1,
        "PRICE_HIGH_PREM_COEF": 0.6,
        "PRICE_MID_COEF": 0.2,
    },

    # ---- 속성 가중치 계산 시 가산치 ----
    "ATTR_WEIGHTS": {
        "DC_CORE": 1.0,
        "DC_CONVENIENCE": 0.8,
        "DC_FRESH_TO_QUALITY": 0.5,
        "DC_FRESH_TO_SAFETY": 0.5,
        "FEAT_HEALTH": 0.6,
        "FEAT_PREMIUM": 0.6,
        "FEAT_CONVENIENCE": 0.4,
        "FEAT_VALUE_PRICE": 0.6,
        "FEAT_VALUE_PS": 0.4,
        "FEAT_TASTE": 0.4,
        "EXTRA_ATTR_BONUS": 0.8,
    },

    # ---- 구매주기 → 월 횟수 추정 ----
    "PURCHASE_CYCLE_MAP": [
        ("주4~6회", 20), ("주4-6회", 20),
        ("주2~3회", 10), ("주2-3회", 10),
        ("주1회", 4),
        ("2주일에1회", 2), ("격주", 2),
        ("1달에1회", 1), ("월1회", 1),
    ],
}


## 1) 유틸/정규화

In [9]:
import re
from typing import Any, Dict, List

def norm_text(s: Any) -> str:
    return re.sub(r"\s+", "", str(s)).strip().lower()

def price_bucket(price: float) -> str:
    low_max = CONFIG["PRICE_BUCKET"]["LOW_MAX"]
    high_min = CONFIG["PRICE_BUCKET"]["HIGH_MIN"]
    try:
        p = float(price)
    except Exception:
        p = 0.0
    if p >= high_min: return "high"
    if p <= low_max:  return "low"
    return "mid"

def purchase_cycle_to_monthly_base(cycle: str) -> int:
    if not cycle:
        return 1
    c = norm_text(cycle)
    for key, val in CONFIG["PURCHASE_CYCLE_MAP"]:
        if norm_text(key) in c:
            return val
    return 1

def infer_product_group(p: Dict[str, Any]) -> str:
    name = p.get("product_name", "")
    cat = p.get("category", {}) or {}
    c = norm_text(f"{cat.get('level_1','')} {cat.get('level_2','')} {cat.get('level_3','')}")
    if "요거트" in name or "하이그릭" in name: return "요거트"
    if "라떼" in name or "카페라떼" in name or "바닐라" in name: return "커피음료"
    if "리챔" in name or "오믈레햄" in name or "햄" in name: return "축산캔"
    if "참치액" in name: return "참치액/조미료"
    if "참기름" in name: return "유지류"
    if "참치" in name or "캔" in name or "라이트스탠다드참치" in c: return "참치캔"
    if "조미료" in cat.get("level_2", ""): return "참치액/조미료"
    return "기타"

def normalize_dc_list(dc_list: Any) -> List[str]:
    if not dc_list: return []
    base_set = set()
    for base, syns in CONFIG["DC_NORMALIZE"].items():
        for item in dc_list:
            if not item: continue
            item_clean = str(item).replace(" ", "")
            for syn in syns:
                if item_clean.replace(" ", "").startswith(syn.replace(" ", "")):
                    base_set.add(base)
    return list(base_set)


## 2) 적합도 계산 & 속성 생성

In [10]:
def compute_fit(product: Dict[str, Any], person: Dict[str, Any]) -> float:
    W = CONFIG["FIT_WEIGHTS"]
    ori_scale = CONFIG["ORI_SCALE_MAX"]
    feature_to_ori = CONFIG["FEATURE_TO_ORI"]
    category_map = CONFIG["CATEGORY_MAP"]

    score = 0.0

    # Category affinity
    pg = infer_product_group(product)
    cat_keywords = [norm_text(k) for k in category_map.get(pg, [])]
    top_cats = [norm_text(x) for x in (person.get("shopping_profile", {}).get("top_categories") or []) if x and x != "응답없음"]
    if top_cats:
        if len(top_cats) >= 1 and any(k in top_cats[0] for k in cat_keywords):
            score += W["TOP_CATEGORY_1"]
        if len(top_cats) >= 2 and any(k in top_cats[1] for k in cat_keywords):
            score += W["TOP_CATEGORY_2"]

    # Household match
    tgt = (product.get("targeted_consumer") or "")
    if tgt:
        tgt_tokens = [t.strip() for t in tgt.split(",")]
        hh = (person.get("demographics", {}) or {}).get("household", "") or ""
        for t in tgt_tokens:
            if t and t in hh:
                score += W["HOUSEHOLD_MATCH"]
                break

    # Feature - orientation
    feats = [f.strip() for f in (product.get("feature", "") or "").split(",")]
    ori = (person.get("orientations") or {})
    for f in feats:
        key = feature_to_ori.get(f.strip(), None)
        if key in ori and isinstance(ori[key], (int, float)):
            score += (ori[key] / ori_scale) * W["ORI_FEAT_MATCH"]
        elif key == "taste":
            score += W["TASTE_FALLBACK"]

    # Decision criteria
    dc = normalize_dc_list(person.get("shopping_profile", {}).get("decision_criteria"))
    for base in dc:
        if base in ["맛", "가격", "품질", "안전성"]:
            score += W["DC_CORE"]
        if base in ["조리의 편리성", "구입의 편리성"]:
            score += W["DC_CONVENIENCE"]
        if base == "신선도" and pg in ["축산캔", "참치캔", "요거트", "커피음료"]:
            score += W["DC_FRESHNESS"]

    # Price fit
    pb = price_bucket(product.get("price", 0))
    ps = (ori.get("price_sensitivity", 5) or 5) / ori_scale
    prem = (ori.get("premium", 5) or 5) / ori_scale
    if pb == "low":
        score += W["PRICE_LOW_PS_COEF"] * ps + W["PRICE_LOW_BIAS"]
    elif pb == "high":
        score += W["PRICE_HIGH_PREM_COEF"] * prem
    else:
        score += W["PRICE_MID_COEF"] * (prem + ps)

    return float(score)

def build_attributes(product: Dict[str, Any], person: Dict[str, Any]) -> list:
    import pandas as pd
    A = CONFIG["ATTR_WEIGHTS"]
    ori_scale = CONFIG["ORI_SCALE_MAX"]

    ori = (person.get("orientations") or {})
    dc = normalize_dc_list(person.get("shopping_profile", {}).get("decision_criteria"))
    feats = [f.strip() for f in (product.get("feature", "") or "").split(",")]
    name = product.get("product_name", "")
    pg = infer_product_group(product)

    extra_attrs = []
    if "요거트" in name or pg == "요거트":
        if "유당불내증" in (product.get("targeted_consumer") or ""):
            extra_attrs.append("락토프리/소화")
        if "고단백" in feats:
            extra_attrs.append("고단백")
    if "참치액" in name:
        extra_attrs.extend(["감칠맛"])
        if "500g" in name: extra_attrs.append("용량_500g")
        if "900g" in name: extra_attrs.append("용량_900g")
        if "진" in name:   extra_attrs.append("진한맛")
        if "순" in name:   extra_attrs.append("깔끔한맛")
        if "프리미엄" in name: extra_attrs.append("프리미엄원재료")
    if "참기름" in name:
        extra_attrs.append("고소한맛")
        if "90g" in name:  extra_attrs.append("소용량_90g")
        if "135g" in name: extra_attrs.append("중간용량_135g")
        if "매콤" in name: extra_attrs.append("매콤풍미")
    if "리챔" in name:
        extra_attrs.extend(["간편조리","낮은칼로리"])
        if "200g" in name: extra_attrs.append("소용량_200g")
        if "340g" in name: extra_attrs.append("대용량_340g")
    if "라떼" in name or "카페라떼" in name:
        extra_attrs.append("간편섭취")
        if "바닐라" in name: extra_attrs.append("단맛취향")
        if "유당불내증" in (product.get("targeted_consumer") or ""):
            extra_attrs.append("락토프리/소화")

    attrs = [
        "확인_맛","확인_가격","확인_브랜드","확인_품질","확인_안전성","확인_용량",
        "편의성_조리","편의성_구입","프리미엄지향","건강/영양",
        "가격민감","브랜드충성","다양성추구"
    ] + extra_attrs

    w = {a: 0.0 for a in attrs}
    w["건강/영양"]   += (ori.get("health", 5) or 5) / ori_scale
    w["가격민감"]    += (ori.get("price_sensitivity", 5) or 5) / ori_scale
    w["편의성_조리"] += (ori.get("hmr", 5) or 5) / ori_scale
    w["브랜드충성"]   += (ori.get("brand_loyalty", 5) or 5) / ori_scale
    w["프리미엄지향"] += (ori.get("premium", 5) or 5) / ori_scale
    w["편의성_구입"]  += (ori.get("convenience", 5) or 5) / ori_scale
    w["다양성추구"]   += (ori.get("variety_seeking", 5) or 5) / ori_scale

    for base in dc:
        if base == "맛": w["확인_맛"] += A["DC_CORE"]
        if base == "가격": w["확인_가격"] += A["DC_CORE"]
        if base == "품질": w["확인_품질"] += A["DC_CORE"]
        if base == "안전성": w["확인_안전성"] += A["DC_CORE"]
        if base == "조리의 편리성": w["편의성_조리"] += A["DC_CONVENIENCE"]
        if base == "구입의 편리성": w["편의성_구입"] += A["DC_CONVENIENCE"]
        if base == "신선도":
            w["확인_품질"] += A["DC_FRESH_TO_QUALITY"]
            w["확인_안전성"] += A["DC_FRESH_TO_SAFETY"]

    for f in feats:
        f = f.strip()
        if f in ["고단백","건강식품"] and "건강/영양" in w: w["건강/영양"] += A["FEAT_HEALTH"]
        if f in ["프리미엄"] and "프리미엄지향" in w: w["프리미엄지향"] += A["FEAT_PREMIUM"]
        if f in ["간편함"]:
            w["편의성_구입"] += A["FEAT_CONVENIENCE"]
            w["편의성_조리"] += A["FEAT_CONVENIENCE"]
        if f in ["가성비"]:
            w["확인_가격"] += A["FEAT_VALUE_PRICE"]
            w["가격민감"] += A["FEAT_VALUE_PS"]
        if f in ["매콤한맛","고소한맛","감칠맛"] and "확인_맛" in w:
            w["확인_맛"] += A["FEAT_TASTE"]

    for a in extra_attrs:
        w[a] = w.get(a, 0.0) + A["EXTRA_ATTR_BONUS"]

    import pandas as pd
    w_series = pd.Series(w)
    w_series = w_series[w_series > 0]
    if len(w_series) < CONFIG["ATTR_MIN"]:
        need = CONFIG["ATTR_MIN"] - len(w_series)
        zeros = pd.Series({k: 0.0001 for k in w if k not in w_series})
        w_series = pd.concat([w_series, zeros.iloc[:need]])
    w_series = (w_series / w_series.sum()).sort_values(ascending=False)
    w_series = w_series.iloc[:CONFIG["ATTR_CAP"]]
    return [{"name": k, "weight": round(float(v), 4)} for k, v in w_series.items()]


## 3) 페르소나 구축/출력 함수

In [11]:
import json, zipfile, re
from pathlib import Path

def slugify(name: str) -> str:
    s = re.sub(r"\s+", "_", name.strip())
    s = re.sub(r"[^0-9A-Za-z가-힣_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "product"

def build_personas(products: list, people: list, topk: int) -> dict:
    out = {}
    for p in products:
        scored = [(person, compute_fit(p, person)) for person in people]
        scored.sort(key=lambda x: x[1], reverse=True)

        selected, used = [], set()
        for person, s in scored:
            seg = person.get("segment_id")
            if seg in used: continue
            selected.append((person, s))
            used.add(seg)
            if len(selected) >= topk: break

        persona_list = []
        for person, s in selected:
            persona_list.append({
                "product_name": p.get("product_name"),
                "product_group": infer_product_group(p),
                "price": p.get("price"),
                "targeted_consumer": p.get("targeted_consumer"),
                "segment_id": person.get("segment_id"),
                "segment_label": person.get("segment_label"),
                "source_persona_id": person.get("persona_id"),
                "demographics": person.get("demographics"),
                "shopping_profile": person.get("shopping_profile"),
                "orientations": person.get("orientations"),
                "fit_score": round(float(s), 3),
                "monthly_base_freq": purchase_cycle_to_monthly_base(person.get("shopping_profile",{}).get("purchase_cycle")),
                "attributes": build_attributes(p, person),
            })
        out[p.get("product_name")] = persona_list
    return out

def write_outputs(personas: dict):
    pconf = CONFIG["PATHS"]
    out_dir = Path(pconf["OUT_DIR"]).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) 통합 JSON
    out_json = out_dir / pconf["OUTPUT_JSON"]
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(personas, f, ensure_ascii=False, indent=2)

    # 2) per-product JSONs
    per_dir = out_dir / pconf["PER_PRODUCT_DIRNAME"]
    per_dir.mkdir(parents=True, exist_ok=True)
    manifest, files = [], []
    for product_name, plist in personas.items():
        slug = slugify(product_name)
        ppath = per_dir / f"{slug}.json"
        with open(ppath, "w", encoding="utf-8") as f:
            json.dump({product_name: plist}, f, ensure_ascii=False, indent=2)
        manifest.append({"product_name": product_name, "file": str(ppath)})
        files.append(ppath)
    with open(per_dir / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    # 3) zip
    zip_path = out_dir / pconf["SPLIT_ZIP"]
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in files:
            zf.write(p, arcname=p.name)

    return {"out_json": str(out_json), "per_dir": str(per_dir), "zip": str(zip_path)}


## 4) 실행

In [12]:
from pathlib import Path
import json

def run_pipeline():
    pconf = CONFIG["PATHS"]
    data_dir = Path(pconf["DATA_DIR"]).resolve()
    in_prod = data_dir / pconf["PRODUCTS"]
    in_people = data_dir / pconf["PEOPLE"]

    with open(in_prod, "r", encoding="utf-8") as f:
        products = json.load(f)
    with open(in_people, "r", encoding="utf-8") as f:
        people = json.load(f)

    personas = build_personas(products, people, topk=CONFIG["TOP_K"])
    outs = write_outputs(personas)

    total_products = len(personas)
    total_personas = sum(len(v) for v in personas.values())
    print(f"[OK] products={total_products}, personas={total_personas}")
    print(f"[OUT] {outs['out_json']}")
    print(f"[OUT] {outs['per_dir']}")
    print(f"[OUT] {outs['zip']}")

# 실행
run_pipeline()


[OK] products=15, personas=90
[OUT] C:\output\product_personas.json
[OUT] C:\output\personas
[OUT] C:\output\product_personas_split.zip
